In [ ]:
DATASET = "prev40_dim4_cont_new"
EXP = "cont_dist_test"
FEATURE_MAP = "specs/mres_scm_4.json"

N_BOOT = 20
N_PERM = 20
SUB_FRAC = 0.8
SEED = 4

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif

PROJECT_ROOT = ".."

rng = np.random.default_rng(seed=SEED)

In [ ]:
population_df = pd.read_csv(f"{PROJECT_ROOT}/data/{EXP}/{DATASET}/population.csv")

with open(f"{PROJECT_ROOT}/{FEATURE_MAP}", 'r') as f:
  feature_map = json.load(f)

In [ ]:
soc_features = [f['name'] for f in feature_map['soc']]
bio_features = [f['name'] for f in feature_map['bio']]
ind_features = [f['name'] for f in feature_map['ind']]
unbiased_features = soc_features + bio_features + ind_features
biased_features = [f"obs_{f['name']}"for f in feature_map['soc']]
features = unbiased_features + biased_features
pathways = ['soc pre-bias'] * len(feature_map['soc']) + ['bio'] * len(feature_map['bio']) + ['ind'] * len(feature_map['ind']) + ['soc post-bias'] * len(feature_map['soc'])
dtypes = {f['name']: f['type'] for f in feature_map['soc'] + feature_map['bio'] + feature_map['ind']}
discrete_mask = np.array([dtypes[f] in ("binary", "categorical") for f in unbiased_features + soc_features])

X = population_df[features].to_numpy()
s = population_df["S"].to_numpy()

print(f"{len(features)} features | {discrete_mask.sum()} discrete, {(~discrete_mask).sum()} continuous")

In [ ]:
n_pop = len(population_df)
n_sub = int(SUB_FRAC * n_pop)
boot_scores = np.empty((N_BOOT, len(features)))

for i in range(N_BOOT):
  idx = rng.choice(n_pop, size=n_sub, replace=False)
  boot_scores[i] = mutual_info_classif(
    X[idx], s[idx], discrete_features=discrete_mask, n_neighbors=3, random_state=SEED + i
  )

In [ ]:
s_shuffled = np.tile(s, (N_PERM, 1))
rng.permuted(s_shuffled, axis=1, out=s_shuffled)

null_scores = np.stack([
    mutual_info_classif(X, s_shuffled[i], discrete_features=discrete_mask,
                         n_neighbors=3, random_state=SEED)
    for i in range(N_PERM)
])
noise_floor = null_scores.max(axis=0)

In [ ]:
mi_df = pd.DataFrame({
    "feature": features,
    "pathway": pathways,
    "mi boot mean": boot_scores.mean(axis=0),
    "mi boot low": np.percentile(boot_scores, 2.5, axis=0),
    "mi boot high": np.percentile(boot_scores, 97.5, axis=0),
    "noise_floor": noise_floor
}).sort_values(["mi boot mean"], ascending=[False]).reset_index(drop=True)

mi_df["robustly_above_floor"] = mi_df["mi boot low"] > mi_df["noise_floor"]

mi_df

In [ ]:
ind_leak = mi_df.query("pathway == 'ind' and robustly_above_floor")
if len(ind_leak):
    print("WARNING: ind features with MI above noise floor (possible leakage from S):")
    print(ind_leak)
else:
    print("OK: all ind features are at/below the permutation noise floor.")

fig, ax = plt.subplots(figsize=(8, 0.35 * len(mi_df) + 1))
palette = {"bio": "tab:blue", "soc pre-bias": "tab:pink", "soc post-bias": "tab:red", "ind": "tab:gray"}
sns.barplot(data=mi_df, y="feature", x="mi boot low",
            hue="pathway", palette=palette, dodge=False, ax=ax)
# per-feature noise floor markers
ax.scatter(mi_df["noise_floor"], np.arange(len(mi_df)), color="black", marker="|", s=200,
           label="noise floor", zorder=5)
ax.set_xlabel("Estimated MI(X, S) (nats)")
ax.set_ylabel("")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()